**VRP**

The VRP is the difference between the impled volatility and the shifted realized volatility.

VRP = VIX − Realized Volatility



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the master dataset
df = pd.read_csv('../src/data/master_dataset.csv', index_col='Date', parse_dates=True)

# 2. Compute daily Volatility Risk Premium (VRP)
df['vrp'] = df['vix_decimal'] - df['forward_realized_vol']

# 3. Calculate complete lifetime premium summary values
mean_vrp = df['vrp'].mean()
median_vrp = df['vrp'].median()
win_rate = (df['vrp'] > 0).mean() * 100

print("=== LIFETIME PREMIUM SUMMARY STATISTICS ===")
print(f"Mean VRP:            {mean_vrp:.4f} ({mean_vrp * 100:.2f}%)")
print(f"Median VRP:          {median_vrp:.4f} ({median_vrp * 100:.2f}%)")
print(f"Percentage VRP > 0:  {win_rate:.2f}%")
print("===========================================\n")

# 4. Isolate extreme negative anomalies (deepest inversions)
print("=== TOP 5 DEEPEST PREMIUM INVERSIONS ===")
print(df.sort_values(by='vrp')[['vix_decimal', 'forward_realized_vol', 'vrp']].head(5))
print("========================================\n")

# 5. Plot VRP time series with static zero baseline
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['vrp'], color='purple', label='Volatility Risk Premium (VRP)', alpha=0.7)
plt.axhline(0, color='red', linestyle='--', linewidth=2, label='Zero Baseline')
plt.title('Volatility Risk Premium (VRP) Over Time')
plt.ylabel('VRP (Decimal)')
plt.xlabel('Date')
plt.legend()
plt.show()

In [ ]:
# 1. Plot Frequency Distribution Histogram of VRP
plt.figure(figsize=(10, 6))
plt.hist(df['vrp'], bins=100, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(mean_vrp, color='red', linestyle='--', linewidth=2, label=f'Mean VRP: {mean_vrp*100:.2f}%')
plt.axvline(median_vrp, color='green', linestyle='-', linewidth=2, label=f'Median VRP: {median_vrp*100:.2f}%')
plt.title('Volatility Risk Premium (VRP) Distribution')
plt.xlabel('VRP (Decimal)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# 2. Segment by Year & Plot Historical Bar Chart
df['Year'] = df.index.year
annual_vrp = df.groupby('Year')['vrp'].mean()

plt.figure(figsize=(12, 5))
annual_vrp.plot(kind='bar', color='teal', edgecolor='black')
plt.axhline(0, color='red', linestyle='-', linewidth=1)
plt.title('Annual Mean Volatility Risk Premium')
plt.ylabel('Mean VRP (Decimal)')
plt.xlabel('Year')
plt.xticks(rotation=45)
plt.show()

# 3. Identify the Top 3 Worst-Performing Months in History (by mean monthly VRP)
df['YearMonth'] = df.index.to_period('M')
monthly_vrp = df.groupby('YearMonth')['vrp'].mean()

print("=== TOP 3 WORST MONTHS IN HISTORY (Average Monthly VRP) ===")
print(monthly_vrp.nsmallest(3))
print("==========================================================")

**Why does VRP Exist**

VIX is based on option prices, and investors and portfolio managers use options as insurance to protect their portfolios. Investors are often biased by Loss Aversion - they feel the pain of a loss twice as much as they feel the joy of an equivalent gain. Prospect Theory helps explain why they willingly overpay for volatility insurance to avoid ruin.